# You can't max every -ility at once

> 📓 *Companion to* **Modern Agentic AI Engineer** · Ch 27 §§27.3–27.6 · type: concept-lab

**The promise:** by the end you can *feel* — not just recite — that pushing one quality attribute costs another, and you can run the book's 4-step trade-off method on a concrete choice to a defensible, reversible decision.

This notebook runs **fully offline, free, and deterministic**. No API key, no model calls — just a tiny synthetic scoring model so the trade-off is *visible*.

## 🧠 Why this matters

Functional requirements say what a system *does*. The **quality attributes** — the "-ilities" (scalability, reliability, maintainability, performance, security, cost, observability) — say how *well* it must do it, and that is where architecture is actually decided. You don't choose an architecture for what it computes; you choose it for the qualities it must exhibit (§27.3).

Here is the part that only *clicks* when you watch it happen: you cannot maximize all the -ilities at once. Pushed far enough, every -ility costs another — extreme scalability hurts simplicity and cost; maximal security hurts usability and speed. An architecture is a **deliberate ranking** of the -ilities for *one* system, not a quest to max them all. The rest of this notebook makes that ranking tangible and then gives you the method to do it on purpose (§27.6).

## Objectives & prereqs

**By the end you can:**
- Read a Pareto frontier and explain "at the expense of what?" from a picture, not a slogan.
- Predict which architectural **style** fits a ranked set of forces, then check it against the book's §27.4 table.
- Run the book's **4-step trade-off method** (name forces → generate options → score → decide, favoring the *reversible* option when close).

**Prereqs:** Chapters 24–26 read (a concrete backend to reason about). No special tooling.

**Packages:** Python standard library only. No `requirements.txt` extras, no network, no key.

In [ ]:
# Setup — imports, env, and the MOCK switch.
import os
import random
from itertools import product

from dotenv import load_dotenv

load_dotenv()  # reads a local .env if present; never hardcode keys

# This concept-lab is pure local computation — there is nothing to call. MOCK stays the
# project-wide default (1) so the notebook behaves identically in CI and for every reader,
# with or without a key. We keep the switch for consistency with the rest of the course.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

# Seed everything stochastic so the numbers below are reproducible run-to-run.
random.seed(27)

print(f"MOCK mode: {MOCK}  (offline, free, deterministic — no model calls in this notebook)")

## A tiny model where one -ility costs another

Imagine a request pipeline. You can add **replicas + redundant health checks** to push *reliability* up. Each unit you add genuinely improves a `reliability` score — but it also raises `cost` (more machines) and adds a little `latency` (an extra hop, a check). That is the whole game in miniature: a knob that helps one -ility and hurts others.

The model below is deliberately synthetic — numbers, not data. Diminishing returns on reliability (each replica helps less than the last) and roughly linear cost are baked in, because that is how it behaves in the real world.

In [ ]:
def score_pipeline(replicas: int) -> dict:
    """A toy scoring model for one architectural knob: number of replicas.

    Returns three normalized 0–100 scores. Reliability has *diminishing returns*
    (each replica helps less); cost and latency get steadily worse. Nothing here
    is empirical — it is a deliberately simple shape so the trade-off is visible.
    """
    # Reliability: approaches 100 asymptotically — the first replica matters most.
    reliability = 100 * (1 - 0.55 ** replicas)
    # Cost: each replica adds a roughly fixed amount (lower score = worse/more expensive).
    cost = max(0.0, 100 - 14 * (replicas - 1))
    # Latency budget: extra hops/checks erode it slightly and with mild compounding.
    latency = max(0.0, 100 - 6 * (replicas - 1) ** 1.3)
    return {"replicas": replicas, "reliability": reliability, "cost": cost, "latency": latency}


configs = [score_pipeline(r) for r in range(1, 9)]
print(f"{'replicas':>8}  {'reliability':>11}  {'cost':>6}  {'latency':>7}")
for c in configs:
    print(f"{c['replicas']:>8}  {c['reliability']:>11.1f}  {c['cost']:>6.1f}  {c['latency']:>7.1f}")

**What you just saw.** Going from 1 to 2 replicas buys a *huge* jump in reliability; going from 7 to 8 barely moves it — but cost and latency keep degrading at the same rate the whole time. Somewhere in that table the marginal reliability stops being worth the marginal cost. The architecture question is: *where?* — and that depends on which -ility you ranked first.

## 🔮 Predict: where does the curve bend?

Before you run the next cell, **predict**: if you plot *reliability* (higher is better) against *cost* (where a higher score means cheaper), which configurations sit on the **Pareto frontier** — the set where you can't improve one score without giving up the other? And which configs are simply **dominated** (something else beats them on *both*)?

Write down how many of the 8 configs you think are on the frontier, then run it.

In [ ]:
def pareto_frontier(points: list, keys: tuple) -> set:
    """Indices of points not dominated on *all* of `keys` (higher = better for each)."""
    frontier = set()
    for i, p in enumerate(points):
        dominated = any(
            j != i
            and all(points[j][k] >= p[k] for k in keys)
            and any(points[j][k] > p[k] for k in keys)
            for j in range(len(points))
        )
        if not dominated:
            frontier.add(i)
    return frontier


front = pareto_frontier(configs, ("reliability", "cost"))

# A no-dependency ASCII scatter so the frontier is *visible* without matplotlib.
print("reliability (→ right = better)   vs   cost-score (↑ up = cheaper)\n")
rows = 12
for row in range(rows, -1, -1):
    lo, hi = row / rows * 100 - 4, row / rows * 100 + 4
    line = []
    label = f"{row / rows * 100:5.0f} |"
    cells = [' '] * 52
    for i, c in enumerate(configs):
        if lo <= c['cost'] < hi:
            col = min(51, int(c['reliability'] / 100 * 51))
            cells[col] = '*' if i in front else 'o'
    print(label + ''.join(cells))
print("      +" + "-" * 52)
print("       0" + " " * 44 + "100  reliability\n")
print(f"frontier configs (replicas): {sorted(configs[i]['replicas'] for i in front)}")
print("  '*' = on the Pareto frontier, 'o' = dominated (something beats it on BOTH).")

**What you just saw.** Only a handful of configs are on the frontier; the rest are *dominated* — there is strictly no reason to pick them. But notice the frontier itself offers **no single winner**. Moving along it, you trade reliability for cost point-for-point. The picture *is* the lesson: "at the expense of what?" is not a rhetorical question, it is the literal shape of the curve. Choosing a point requires a *ranking* you bring from outside the model.

## 🔮 Predict: which style fits these forces?

Now zoom out from one knob to the system shape. The book's §27.4 lists five styles. For the scenario below, **predict which style wins** before revealing the guidance:

> *Two engineers, pre-launch product, requirements still shifting. You need fast iteration and one simple deploy, but you want clean internal seams so you can split things out later if a real force shows up.*

Pick one: `monolith` / `modular-monolith` / `microservices` / `event-driven` / `serverless`. Then run the cell to compare against the book's table.

In [ ]:
# The book's §27.4 styles table, verbatim in spirit: good-for and main-cost.
STYLES = {
    "monolith": {
        "good_for": "Small teams, early products, simplicity, speed of iteration",
        "main_cost": "Hard to scale teams/parts independently",
    },
    "modular-monolith": {
        "good_for": "Most systems: clear modules, one deploy — the pragmatic default",
        "main_cost": "Discipline to keep modules truly separate",
    },
    "microservices": {
        "good_for": "Many teams, independent scaling/deploy, fault isolation",
        "main_cost": "Operational + distributed-systems complexity",
    },
    "event-driven": {
        "good_for": "Decoupling, async workflows, audit trails, reactive flows",
        "main_cost": "Harder to trace, reason about, and debug",
    },
    "serverless": {
        "good_for": "Spiky/low traffic, minimal ops, pay-per-use, glue",
        "main_cost": "Cold starts, vendor lock-in, limits",
    },
}

book_pick = "modular-monolith"
for name, info in STYLES.items():
    mark = "  <- book's call" if name == book_pick else ""
    print(f"{name:18s} good for: {info['good_for']}")
    print(f"{'':18s} main cost: {info['main_cost']}{mark}\n")

print("The book: start with a MODULAR MONOLITH and extract services only when a real")
print("force (team scale, independent scaling, fault isolation) demands it (§27.4).")

**What you just saw.** If you predicted `microservices` because it is the "serious" answer, that is exactly the cargo-cult the book warns about (§27.4): you would inherit network failures, eventual consistency, and distributed debugging to solve a problem two pre-launch engineers don't have. The forces here — tiny team, shifting requirements, speed — rank *maintainability* and *speed of iteration* on top, and the modular monolith serves those at the lowest cost while keeping the seams that let you split later.

## The 4-step trade-off method, as code

§27.6: there is no "best", only *best for this context*. The method is deliberate:

1. **Name the forces** — the top -ilities, *ranked*, plus real constraints.
2. **Generate options** — two or three viable approaches (if you see only one, you haven't understood the problem).
3. **Compare against the forces** — score each option on the ranked -ilities; make the cost explicit.
4. **Decide and record why** — and when two are close, pick the more **reversible** one (the §27.6 tip).

Below, that method is a tiny structured exercise: a weighted score over ranked forces, with an explicit reversibility tie-break.

In [ ]:
# Step 1: name the forces, ranked. Weights fall off so rank-1 dominates rank-3.
ranked_forces = ["maintainability", "cost", "reliability"]  # most -> least important
weights = {force: w for force, w in zip(ranked_forces, [3, 2, 1])}

# Step 2: generate options (two viable shapes for the agent platform). Each is scored
# 1-5 on every ranked force, plus a 'reversible' flag for the tie-break in step 4.
options = {
    "modular-monolith": {"maintainability": 5, "cost": 5, "reliability": 3, "reversible": True},
    "microservices":    {"maintainability": 2, "cost": 2, "reliability": 4, "reversible": False},
}


# Step 3: compare — weighted score against the ranked forces.
def weighted_score(option: dict) -> int:
    return sum(weights[f] * option[f] for f in ranked_forces)


scores = {name: weighted_score(opt) for name, opt in options.items()}
print("forces (ranked):", " > ".join(ranked_forces))
print(f"weights:          {weights}\n")
for name, opt in options.items():
    detail = "  ".join(f"{f}={opt[f]}" for f in ranked_forces)
    print(f"{name:18s} score={scores[name]:3d}   {detail}   reversible={opt['reversible']}")

# Step 4: decide — highest score wins; if within 10%, prefer the reversible option.
ranked = sorted(scores, key=scores.get, reverse=True)
top, runner = ranked[0], ranked[1]
close = (scores[top] - scores[runner]) <= 0.10 * scores[top]
if close and options[runner]["reversible"] and not options[top]["reversible"]:
    decision = runner
    why = "scores were close, so the tie-break favored the more REVERSIBLE option"
else:
    decision = top
    why = "it wins on the ranked forces" + (
        " (and is also the more reversible option)" if options[top]["reversible"] else ""
    )
print(f"\nDECISION: {decision}  —  {why}.")

**What you just saw.** The method turns a vibe ("microservices feel scalable") into an *auditable* number: with maintainability and cost ranked first, the modular monolith wins outright — and it is also the reversible choice, so even a near-tie would have broken the same way. The point isn't the exact weights; it's that the forces are named, the options are plural, the cost is explicit, and the *why* is recorded. That record is the ADR you'll write in the next notebook.

## ⚠️ Pitfall: optimizing one -ility in isolation

The rookie move — the one the Pareto curve above makes concrete — is to chase a single quality attribute with no "at the expense of what?":

In [ ]:
# The rookie approach: "make it reliable!" — crank replicas to the max.
rookie = max(configs, key=lambda c: c["reliability"])
print("ROOKIE  'make it reliable!' — picks the max-reliability config, blind to the rest:")
print(f"  replicas={rookie['replicas']}  reliability={rookie['reliability']:.1f}  "
      f"cost={rookie['cost']:.1f}  latency={rookie['latency']:.1f}")
print(f"  -> bought {rookie['reliability'] - configs[1]['reliability']:.1f} extra reliability")
print(f"     over 2 replicas, at {configs[1]['cost'] - rookie['cost']:.1f} cost-score and "
      f"{configs[1]['latency'] - rookie['latency']:.1f} latency-score lost.\n")

# The senior approach: ask where the *marginal* reliability stops being worth it.
best_marginal, prev = None, None
for c in configs:
    if prev is not None:
        gain = c["reliability"] - prev["reliability"]
        if gain < 5:  # less than 5 points of reliability per added replica = stop
            best_marginal = prev
            break
    prev = c
print("SENIOR  'what meets the ranked -ilities most cheaply?' — stops at diminishing returns:")
print(f"  replicas={best_marginal['replicas']}  reliability={best_marginal['reliability']:.1f}  "
      f"cost={best_marginal['cost']:.1f}  latency={best_marginal['latency']:.1f}")

**The fix.** "Make it scalable!" / "make it reliable!" is an instruction with no budget. Always pair an -ility with the question "at the expense of what?" and stop where the marginal gain stops paying for the marginal cost. The senior config buys nearly all the reliability for a fraction of the cost and latency hit — because it was chosen against a *ranking*, not in isolation (§27.3 pitfall).

## 🎯 Senior lens

The architect's *first* question is "what's the simplest thing that meets the -ilities?" — not the last. Microservices are the most cargo-culted decision in the field: teams adopt them for résumé reasons or because a famous company did, and inherit a distributed system (network failures, eventual consistency, deployment orchestration, distributed debugging — Chapter 29) to solve a problem they didn't have. The instinct this notebook trains is the opposite: start as simple as the requirements allow, keep clean module boundaries *inside* a monolith, and split out a service only when a concrete force makes the complexity pay. The Pareto curve is the discipline made visual — every gain has a price, so name what you're buying and what you're spending.

## Recap

- The **-ilities** say how *well* a system works; you can't max them all — architecture is a deliberate **ranking** (§27.3).
- A **Pareto frontier** makes "at the expense of what?" literal: along it, every gain in one score costs another.
- The pragmatic default style is a **modular monolith**; extract services only when a real force demands it (§27.4).
- The **4-step method** — name (ranked) forces → generate options → score → decide — turns a vibe into an auditable choice (§27.6).
- When two options are close, pick the **more reversible** one; cheap-to-undo decisions let you learn from production.

## Exercises

Each exercise *changes* something and asks you to predict the result before running.

1. **Re-rank the forces.** In the 4-step cell, change `ranked_forces` so *reliability* is rank-1 and *cost* is rank-3. Predict whether the decision flips to `microservices` before you run it. Why does (or doesn't) it?
2. **Add a third option.** Add a `serverless` option to the `options` dict with your own 1–5 scores and a `reversible` flag. Predict where it lands in the ranking, then check.
3. **Change the knob's shape.** In `score_pipeline`, make cost degrade *faster* (e.g. `-22 * (replicas - 1)`). Predict how the Pareto frontier shrinks, then re-run the scatter cell.
4. **Your own system.** Write down the top three -ilities, *ranked*, for a system you know, and one force that would justify extracting a service from a modular monolith.

In [ ]:
# Exercise scratch space — your code here.


In [ ]:
# Exercise scratch space — your code here.


## Next

- **Next notebook:** [`27-02-adr-and-c4-worksheet.ipynb`](27-02-adr-and-c4-worksheet.ipynb) — you ranked the -ilities and chose a style; now *record the why*. You'll write a full ADR and a C4 sketch for the capstone.
- **Book:** revisit §27.3 (the -ilities), §27.4 (styles), and §27.6 (trade-off analysis and the reversibility tip).
- **Templates this feeds:** the ranked-forces + scored-options table you just built is the trade-off section of [`templates/system-design-doc/`](../../templates/system-design-doc/); the recorded decision becomes an [`templates/adr-template/`](../../templates/adr-template/) entry in the next notebook.
- **Capstone:** this judgment is the architectural narrative behind [`capstone-project/`](../../capstone-project/) (Appendix C) — no code, but it shapes everything that has code.